In [2]:
!pip install dlt[bigquery]

In [3]:
import itertools, os
from tqdm import tqdm_notebook
year = ['2019','2020']
months = list(map(lambda x: '0'+str(x) if x<10 else str(x), list(range(1,13))))
year_months = set(itertools.product(months, year))

In [ ]:
URLS = []
for month, year in year_months:
    YELLOW_URL = f"https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_{year}-{month}.csv.gz"
    GREEN_URL = f"https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_{year}-{month}.csv.gz"
    URLS.extend([YELLOW_URL, GREEN_URL])

In [4]:
os.makedirs('data/yellow')
os.makedirs('data/green')

In [ ]:
for URL in tqdm_notebook(URLS):
    dirpath = ['data']
    dirpath.extend(URL.split('/')[-2:])
    dirpath = '/'.join(dirpath)
    os.system(f"wget {URL} -O {dirpath}")


In [ ]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

files = filesystem(bucket_url="data/green", file_glob="*.csv.gz")
files.apply_hints(incremental=dlt.sources.incremental("modification_date"))
reader = (files | read_csv()).with_name("green_tripdata")
pipeline = dlt.pipeline(pipeline_name="green_taxi_datapipe", dataset_name="trips_newdata", destination="bigquery")

info = pipeline.run(reader)
print(info)

In [ ]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

files = filesystem(bucket_url="data/yellow", file_glob="*.csv.gz")
files.apply_hints(incremental=dlt.sources.incremental("modification_date"))
reader = (files | read_csv()).with_name("yellow_tripdata")
pipeline = dlt.pipeline(pipeline_name="yellow_taxi_datapipe", dataset_name="trips_newdata", destination="bigquery")

info = pipeline.run(reader)
print(info)

In [8]:
import itertools, os
from tqdm import tqdm_notebook
year = ['2019']
months = list(map(lambda x: '0'+str(x) if x<10 else str(x), list(range(1,13))))
year_months = set(itertools.product(months, year))

In [9]:
year_months

{('01', '2019'),
 ('02', '2019'),
 ('03', '2019'),
 ('04', '2019'),
 ('05', '2019'),
 ('06', '2019'),
 ('07', '2019'),
 ('08', '2019'),
 ('09', '2019'),
 ('10', '2019'),
 ('11', '2019'),
 ('12', '2019')}

In [6]:
URL_FHV = []
for month, year in year_months:
    FHV_URL = f"https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhv/fhv_tripdata_{year}-{month}.csv.gz"
    URL_FHV.append(FHV_URL)

os.makedirs('data/fhv')

for URL in tqdm_notebook(URL_FHV):
    dirpath = ['data']
    dirpath.extend(URL.split('/')[-2:])
    dirpath = '/'.join(dirpath)
    os.system(f"wget {URL} -O {dirpath}")


C:\Users\satiy\AppData\Local\Temp\ipykernel_5768\875406003.py:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for URL in tqdm_notebook(URL_FHV):


  0%|          | 0/24 [00:00<?, ?it/s]

In [7]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

files = filesystem(bucket_url="data/fhv", file_glob="*.csv.gz")
files.apply_hints(incremental=dlt.sources.incremental("modification_date"))
reader = (files | read_csv()).with_name("fhv_tripdata")
pipeline = dlt.pipeline(pipeline_name="fhv_taxi_datapipe", dataset_name="trips_newdata", destination="bigquery")

info = pipeline.run(reader)
print(info)

C:\Users\satiy\anaconda3\envs\dlt_env\lib\site-packages\google\cloud\bigquery\client.py:595: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


PipelineStepFailed: Pipeline execution failed at stage extract when processing package 1740866003.7777212 with exception:

<class 'dlt.extract.exceptions.ResourceExtractionError'>
In processing pipe fhv_tripdata: extraction of resource fhv_tripdata in generator _read_csv caused an exception: 'utf-8' codec can't decode byte 0xa0 in position 41721: invalid start byte

In [11]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

# Define the filesystem source with CSV files
files = filesystem(bucket_url="data/fhv", file_glob="*.csv.gz")

# Read CSV files and apply incremental loading on "pickup_datetime"
@dlt.resource(
    name="fhv_tripdata", 
    write_disposition="merge",  # Ensures updates instead of duplicates
    primary_key="id",  # Define a primary key for deduplication
    incremental=dlt.sources.incremental("pickup_datetime")  # Incremental based on datetime
)
def read_fhv_tripdata():
    return files | read_csv(encoding="utf-8", encoding_errors="replace")

# Initialize DLT pipeline to BigQuery
pipeline = dlt.pipeline(
    pipeline_name="fhv_taxi_datapipe",
    dataset_name="trips_newdata",
    destination="bigquery"
)

# Run the pipeline with incremental loading
info = pipeline.run(read_fhv_tripdata())

print(info)


C:\Users\satiy\anaconda3\envs\dlt_env\lib\site-packages\google\cloud\bigquery\client.py:595: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


PipelineStepFailed: Pipeline execution failed at stage extract when processing package 1740894284.7101445 with exception:

<class 'dlt.extract.exceptions.ResourceExtractionError'>
In processing pipe _read_csv: extraction of resource _read_csv in generator _read_csv caused an exception: [Errno 2] No such file or directory: 'C:/Users/satiy/Downloads/data_engineering_zoomcamp/04-analytics-engineering/dlt_dataloader/data/fhv/fhv_tripdata_2020-01.csv.gz'